# Eksperimen: Model 1 (Intent) dengan Data Sesi Chat

Model gabungan (`nlu_gabungan.ipynb`) jauh mengungguli pipeline dua model pada penilaian ujung ke ujung: exact match target 0.755 vs 0.497. Ada dua penjelasan yang bersaing:

| Penjelasan | Artinya | Diuji dengan |
| --- | --- | --- |
| **Arsitektur** | Satu encoder untuk dua tugas, dan relasi tidak bergantung pada tebakan intent. | Model gabungan |
| **Data** | Encoder gabungan juga melihat pesan bergaya sesi chat (dataset target), sedangkan Model 1 hanya melihat `dataset_final3.csv`. | **Notebook ini:** Model 1 biasa (satu tugas) yang dilatih dengan data intent **ditambah semua pesan sesi chat** di bagian train dataset target |

Bila pipeline dengan Model 1 baru ini menyusul model gabungan, keunggulan gabungan terutama berasal dari data. Bila tidak, arsitekturnya yang berperan.

Supaya setara dengan run 27 Oktober, data tambahan plan03 **tidak** dipakai di sini. Split, pengaturan IndoBERT, Model 2, dan cara penilaian sama dengan `nlu_baru`, `nlu_target`, dan `nlu_gabungan`.

## Library yang digunakan

In [1]:
# Library standar Python: file, waktu, dan memori.
import gc
import json
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path
from time import perf_counter

# Pengolahan data dan tampilan tabel.
import numpy as np
import pandas as pd
from IPython.display import display

# Pembagian data dan metrik (sama dengan notebook lain).
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# IndoBERT.
import torch
from datasets import Dataset
from safetensors.torch import load_file
from torch import nn
from transformers import (
    AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed,
)

C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pengaturan eksperimen

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `DATASET_INTENT / DATASET_TARGET` | Sumber data. | Sama dengan `nlu_baru` dan `nlu_target` (label sesudah relabel 26 Okt). |
| `MODEL_1_DIR / MODEL_2_DIR / MODEL_GABUNGAN_DIR` | Model pembanding. | Model 1 lama (tanpa data sesi), Model 2, dan model gabungan dari run 27 Okt. |
| `MODEL_SESI_DIR` | Tempat menyimpan model eksperimen. | Folder terpisah; **tidak** menimpa Model 1 yang dipakai game. |
| `LATIH_INDOBERT` | Saklar training. | False: memuat model tersimpan dari `MODEL_SESI_DIR`. |

In [2]:
# Sumber data dan model pembanding.
DATASET_INTENT = Path("data/dataset_final3.csv")
DATASET_TARGET = Path("data/dataset_final3_context_target_pilot25_lengkap.csv")
MODEL_1_DIR = Path("models/notebook_standalone/intent_classifier_transformer")
MODEL_2_DIR = Path("models/target_classifier/target_classifier_transformer")
MODEL_GABUNGAN_DIR = Path("models/gabungan/nlu_gabungan_transformer")

# Model eksperimen disimpan terpisah agar Model 1 yang dipakai game tidak tertimpa.
MODEL_SESI_DIR = Path("models/eksperimen/intent_sesi_transformer")

RANDOM_STATE = 42
JUMLAH_PEMAIN_MINIMUM = 6
LATIH_INDOBERT = True
INTENT_LABELS = ["defend", "neutral", "offend"]
RELASI_LABELS = ["defend", "neutral", "offend", "tidak_ada"]

## Data

Loader, penyamaran nama, teks pasangan, varian nama, dan pembagian data disalin dari `nlu_gabungan.ipynb`, yang sendiri disalin dari `nlu_baru`/`nlu_target`.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `load_clean_nlu_dataset / load_target_dataset` | Membaca dan memvalidasi dataset. | Sama dengan notebook asal. |
| `samarkan_nama / buat_teks_pasangan / bentuk_pasangan` | Teks pasangan untuk Model 2 dan model gabungan. | `intent_awalan` menentukan awalan intent pipeline. |
| `varian_nama_intent` | Kondisi game untuk intent. | Nama diganti tiga set nama yang dikenal model; peluang dirata-ratakan. |

In [3]:
# Membaca CSV intent dan membersihkannya (sama dengan nlu_baru).
def load_clean_nlu_dataset(dataset_path):
    raw = pd.read_csv(dataset_path)
    clean = raw[["teks_chat", "label_intent"]].dropna().copy()
    clean["teks_chat"] = clean["teks_chat"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    clean["label_intent"] = clean["label_intent"].astype(str).str.strip().str.lower()
    clean = clean.loc[(clean["teks_chat"] != "") & (clean["label_intent"] != "")].drop_duplicates(["teks_chat", "label_intent"])
    ganda = clean.groupby("teks_chat")["label_intent"].nunique()
    clean = clean.loc[~clean["teks_chat"].isin(ganda[ganda > 1].index)]
    return clean.reset_index(drop=True)


# Membaca CSV target dan mengubah kolom JSON (aturan sama dengan nlu_target; data sudah lolos validasinya).
def load_target_dataset(dataset_path):
    data = pd.read_csv(dataset_path, keep_default_na=False)
    for column in ["chat_sebelumnya", "daftar_pemain", "target"]:
        data[column] = data[column].apply(json.loads)
    data["teks_chat"] = data["teks_chat"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    data["label_intent"] = data["label_intent"].astype(str).str.strip().str.lower()
    return data.reset_index(drop=True)


TOKEN_KANDIDAT, TOKEN_PENGIRIM, TOKEN_PEMAIN, TOKEN_TIDAK_DIKETAHUI = "KANDIDAT_X", "PENGIRIM_X", "PEMAIN_X", "TIDAK_DIKETAHUI"
KATA_GANTI = {"aku", "saya", "gw", "gue", "gua", "dia", "kamu", "kau", "lu", "lo", "elu",
              "anda", "mereka", "kalian", "kita", "kami", "semua", "si"}
ISTILAH_GAME = {"hitman", "spy", "stalker", "civilian", "hostage", "gag", "order", "guard", "peek", "vote"}
AKHIRAN_NAMA = ("nya", "lah", "kah", "pun", "mu", "ku")
NAMA_SEMU = ["Agus", "Bella", "Candra", "Desi", "Erik", "Fitri", "Galang", "Hesti", "Irwan", "Jessi"]
NAMA_KANONIK = (
    ("Raka", "Lala", "Sari", "Naya", "Dito", "Kiki", "Fajar", "Tio", "Bima", "Wawan"),
    ("Dito", "Kiki", "Fajar", "Tio", "Bima", "Wawan", "Raka", "Lala", "Sari", "Naya"),
    ("Bima", "Wawan", "Raka", "Lala", "Sari", "Naya", "Dito", "Kiki", "Fajar", "Tio"),
)


def build_common_words(source_path, min_count=3):
    counts, total, kapital = Counter(), Counter(), Counter()
    for text in pd.read_csv(source_path)["teks_chat"].dropna().astype(str):
        counts.update(set(re.findall(r"\w+", text.casefold())))
        for index, match in enumerate(re.finditer(r"\w+", text)):
            if index == 0 or text[:match.start()].rstrip().endswith((".", "!", "?")):
                continue
            key = match.group(0).casefold()
            total[key] += 1
            kapital[key] += match.group(0)[:1].isupper()
    return frozenset(w for w, c in counts.items() if c >= min_count and len(w) >= 2 and kapital[w] <= total[w] * 0.5)


@lru_cache(maxsize=None)
def _nama_cocok(word, name):
    w, n = word.casefold(), name.casefold()
    if len(w) < 3:
        return False
    if w in n:
        return True
    sisa = w[len(n):] if w.startswith(n) else None
    return sisa is not None and (sisa in AKHIRAN_NAMA or set(sisa) == {n[-1]})


def samarkan_nama(text, daftar_pemain, kandidat, pengirim, common_words):
    lowered = {name.casefold(): name for name in daftar_pemain}

    def token_for(name):
        if name == kandidat:
            return TOKEN_KANDIDAT
        return TOKEN_PENGIRIM if name == pengirim else TOKEN_PEMAIN

    def replace(match):
        word = match.group(0)
        key = word.casefold()
        if key in lowered:
            return token_for(lowered[key])
        if key in KATA_GANTI or key in ISTILAH_GAME or key in common_words:
            return word
        matches = [name for name in daftar_pemain if _nama_cocok(word, name)]
        return token_for(matches[0]) if len(matches) == 1 else word

    return re.sub(r"\w+", replace, text)


def buat_teks_pasangan(pengirim, teks_chat, chat_sebelumnya, daftar_pemain, kandidat, common_words, label_intent=None):
    def peran(name):
        if name == "tidak_diketahui":
            return TOKEN_TIDAK_DIKETAHUI
        if name == kandidat:
            return TOKEN_KANDIDAT
        return TOKEN_PENGIRIM if name == pengirim else TOKEN_PEMAIN

    bagian = []
    for pesan in reversed(chat_sebelumnya):
        relasi = ", ".join(f"{t['relasi']} {peran(t['pemain'])}" for t in pesan["target"])
        anotasi = f"{pesan['label_intent']}: {relasi}" if relasi else pesan["label_intent"]
        bagian.append(f"{peran(pesan['pengirim'])}: {samarkan_nama(pesan['teks_chat'], daftar_pemain, kandidat, pengirim, common_words)} [{anotasi}]")
    sekarang = samarkan_nama(teks_chat, daftar_pemain, kandidat, pengirim, common_words)
    awal = f"intent {label_intent} || " if label_intent else ""
    return f"{awal}pesan sekarang {peran(pengirim)}: {sekarang} || konteks: {' | '.join(bagian) if bagian else 'tidak ada'}"


def bentuk_pasangan(data, common_words, jumlah_pemain_minimum=0, intent_awalan=None):
    rows = []
    for row_id, row in data.iterrows():
        gold = {t["pemain"]: t["relasi"] for t in row["target"] if t["pemain"] != "tidak_diketahui"}
        kandidat_list = list(row["daftar_pemain"])
        semu = [name for name in NAMA_SEMU if name.casefold() not in {p.casefold() for p in kandidat_list}]
        kandidat_list += semu[:max(0, jumlah_pemain_minimum - len(kandidat_list))]
        awalan = None if intent_awalan is None else intent_awalan[row_id]
        for kandidat in kandidat_list:
            rows.append({"row_id": row_id, "label_intent": row["label_intent"], "kandidat": kandidat,
                         "teks": buat_teks_pasangan(row["pengirim"], row["teks_chat"], row["chat_sebelumnya"],
                                                    row["daftar_pemain"], kandidat, common_words, awalan),
                         "relasi": gold.get(kandidat, "tidak_ada")})
    return pd.DataFrame(rows)


def varian_nama_intent(text, daftar_pemain, common_words):
    lowered = {name.casefold(): name for name in daftar_pemain}

    def pemain(word):
        key = word.casefold()
        if key in lowered:
            return lowered[key]
        if key in KATA_GANTI or key in ISTILAH_GAME or key in common_words:
            return None
        matches = [name for name in daftar_pemain if _nama_cocok(word, name)]
        return matches[0] if len(matches) == 1 else None

    urutan = {}
    for word in re.findall(r"\w+", text):
        nama = pemain(word)
        if nama is not None:
            urutan.setdefault(nama, len(urutan))
    if not urutan:
        return [text]

    def ganti(set_nama):
        def replace(match):
            nama = pemain(match.group(0))
            return match.group(0) if nama is None else set_nama[urutan[nama] % len(set_nama)]
        return re.sub(r"\w+", replace, text)

    return [ganti(set_nama) for set_nama in NAMA_KANONIK]


# Langsung baca data dan siapkan kata umum.
data_intent = load_clean_nlu_dataset(DATASET_INTENT)
data_target = load_target_dataset(DATASET_TARGET)
COMMON_WORDS = build_common_words(DATASET_INTENT)
print("Intent:", len(data_intent), "chat | Target:", len(data_target), "sesi | kata umum:", len(COMMON_WORDS))

Intent: 5611 chat | Target: 775 sesi | kata umum: 1185


### Pembagian Data dan Data Sesi

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `split_intent` | Train/validation/test intent. | Sama dengan `nlu_baru`; **test intent tidak berubah**. |
| `split_target` | Train/validation/test sesi. | Sama dengan `nlu_target`; test target 155 pesan yang sama. |
| `pesan_sesi` | Pesan utama + pesan riwayat dari sesi **train** saja, beserta label intentnya. | Ditambahkan ke train Model 1. Pesan dari sesi validation/test tidak dipakai, dan teks yang sama dengan validation/test intent dibuang. |

In [4]:
# Split intent (nlu_baru) dan split target (nlu_target).
x_pool, x_test, y_pool, y_test = train_test_split(data_intent["teks_chat"], data_intent["label_intent"], test_size=0.2,
                                                  random_state=RANDOM_STATE, stratify=data_intent["label_intent"])
x_train, x_val, y_train, y_val = train_test_split(x_pool, y_pool, test_size=0.15 / 0.8, random_state=RANDOM_STATE,
                                                  stratify=y_pool)


def _split_rows(data, test_size=0.2):
    splitter = StratifiedGroupKFold(n_splits=round(1 / test_size), shuffle=True, random_state=RANDOM_STATE)
    train_index, test_index = next(splitter.split(data, data["case_name"], groups=data["source_group"]))
    return data.iloc[train_index], data.iloc[test_index]


pool_rows, test_rows = _split_rows(data_target)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
i_train, i_val = next(splitter.split(pool_rows, pool_rows["case_name"], groups=pool_rows["source_group"]))
split_target = {"train": pool_rows.iloc[i_train], "validation": pool_rows.iloc[i_val], "test": test_rows}

# Pesan sesi dari bagian train target: pesan utama dan semua pesan riwayat beserta label intentnya.
pesan_sesi = pd.DataFrame(
    [{"teks_chat": r["teks_chat"], "label_intent": r["label_intent"], "sumber": "utama"}
     for _, r in split_target["train"].iterrows()]
    + [{"teks_chat": c["teks_chat"], "label_intent": c["label_intent"], "sumber": "riwayat"}
       for _, r in split_target["train"].iterrows() for c in r["chat_sebelumnya"]])
pesan_sesi["teks_chat"] = pesan_sesi["teks_chat"].str.replace(r"\s+", " ", regex=True).str.strip()

# Buang teks yang ada di validation/test intent atau di pesan utama validation/test target, lalu teks berlabel ganda.
terlarang = set(x_val) | set(x_test) | set(split_target["validation"]["teks_chat"]) | set(split_target["test"]["teks_chat"])
pesan_sesi = pesan_sesi.loc[~pesan_sesi["teks_chat"].isin(terlarang)].drop_duplicates(["teks_chat", "label_intent"])
ganda = pesan_sesi.groupby("teks_chat")["label_intent"].nunique()
pesan_sesi = pesan_sesi.loc[~pesan_sesi["teks_chat"].isin(ganda[ganda > 1].index)]
pesan_sesi = pesan_sesi.loc[~pesan_sesi["teks_chat"].isin(set(x_train))]

x_train_sesi = pd.concat([x_train, pesan_sesi["teks_chat"]], ignore_index=True)
y_train_sesi = pd.concat([y_train, pesan_sesi["label_intent"]], ignore_index=True)
display(pd.DataFrame([
    {"bagian": "train dataset_final3", "chat": len(x_train), **y_train.value_counts().to_dict()},
    {"bagian": "+ pesan sesi (utama)", "chat": int((pesan_sesi["sumber"] == "utama").sum()),
     **pesan_sesi.loc[pesan_sesi["sumber"] == "utama", "label_intent"].value_counts().to_dict()},
    {"bagian": "+ pesan sesi (riwayat)", "chat": int((pesan_sesi["sumber"] == "riwayat").sum()),
     **pesan_sesi.loc[pesan_sesi["sumber"] == "riwayat", "label_intent"].value_counts().to_dict()},
    {"bagian": "train eksperimen", "chat": len(x_train_sesi), **y_train_sesi.value_counts().to_dict()},
    {"bagian": "validation / test (tetap)", "chat": f"{len(x_val)} / {len(x_test)}"},
]).fillna("").style.hide(axis="index"))

bagian,chat,offend,neutral,defend
train dataset_final3,3646,1708.000000,1257.000000,681.000000
+ pesan sesi (utama),479,204.000000,79.000000,196.000000
+ pesan sesi (riwayat),2898,1090.000000,1261.000000,547.000000
train eksperimen,7023,3002.000000,2597.000000,1424.000000
validation / test (tetap),842 / 1123,,,


## Training IndoBERT

Pengaturan **sama persis** dengan Model 1 di `nlu_baru` (cell "Parameter Training IndoBERT"). Yang berbeda hanya data train.

| Nama variabel / proses | Untuk apa? | Arti nilai dan pengaruhnya |
| --- | --- | --- |
| `learning_rates` | 1e-5, 2e-5, 3e-5 | Pemenang dipilih dari macro-F1 validation intent. |
| `epochs=12`, `patience=3`, `threshold=0.001` | Batas epoch dan early stopping | Sama dengan Model 1. |
| `batch 8 × 2`, `max_length=128` | Batch efektif 16 chat | Sama dengan Model 1. |

In [5]:
# Pengaturan Model 1 (nlu_baru).
OPSI = dict(model_name="indobenchmark/indobert-base-p1", learning_rates=(1e-5, 2e-5, 3e-5), epochs=12, batch_size=8,
            eval_batch_size=16, gradient_accumulation_steps=2, max_length=128, weight_decay=0.01, warmup_ratio=0.1,
            dropout=0.1, max_grad_norm=1.0, patience=3, threshold=0.001)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRESISI = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
           else torch.float16 if DEVICE.type == "cuda" else None)
tokenizer = AutoTokenizer.from_pretrained(OPSI["model_name"], local_files_only=True)
id2label = dict(enumerate(INTENT_LABELS))
label2id = {v: k for k, v in id2label.items()}


def dataset_token(texts, labels):
    ds = Dataset.from_dict({"teks": list(texts), "labels": [label2id[y] for y in labels]})
    return ds.map(lambda ex: tokenizer(ex["teks"], truncation=True, max_length=OPSI["max_length"]),
                  batched=True, remove_columns=["teks"])


def compute_metrics(prediksi):
    logits, actual = prediksi
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(actual, pred), "f1_macro": f1_score(actual, pred, average="macro", zero_division=0)}


# Melatih tiap learning rate dari bobot pralatih dan seed yang sama; pemenang dari validation.
if LATIH_INDOBERT:
    train_ds, val_ds = dataset_token(x_train_sesi, y_train_sesi), dataset_token(x_val, y_val)
    collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8 if DEVICE.type == "cuda" else None)
    hasil_trial, mulai = [], perf_counter()
    for nomor, rate in enumerate(OPSI["learning_rates"], start=1):
        set_seed(RANDOM_STATE)
        model = AutoModelForSequenceClassification.from_pretrained(
            OPSI["model_name"], num_labels=3, id2label=id2label, label2id=label2id, hidden_dropout_prob=OPSI["dropout"],
            attention_probs_dropout_prob=OPSI["dropout"], classifier_dropout=OPSI["dropout"], local_files_only=True)
        model.config.nlu_max_length = OPSI["max_length"]
        args = TrainingArguments(
            output_dir=str(MODEL_SESI_DIR.parent / "runs_intent_sesi" / f"trial_{nomor}"), eval_strategy="epoch",
            save_strategy="epoch", logging_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1_macro",
            greater_is_better=True, learning_rate=rate, per_device_train_batch_size=OPSI["batch_size"],
            per_device_eval_batch_size=OPSI["eval_batch_size"], gradient_accumulation_steps=OPSI["gradient_accumulation_steps"],
            num_train_epochs=OPSI["epochs"], weight_decay=OPSI["weight_decay"], warmup_ratio=OPSI["warmup_ratio"],
            lr_scheduler_type="linear", max_grad_norm=OPSI["max_grad_norm"], save_total_limit=1, save_only_model=True,
            report_to="none", seed=RANDOM_STATE, data_seed=RANDOM_STATE, dataloader_num_workers=0,
            use_cpu=DEVICE.type == "cpu", bf16=PRESISI == torch.bfloat16, fp16=PRESISI == torch.float16, optim="adamw_torch")
        trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer,
                          data_collator=collator, compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(OPSI["patience"], OPSI["threshold"])])
        trainer.train()
        hasil_trial.append({"learning_rate": rate, "val_f1_macro": float(trainer.state.best_metric),
                            "checkpoint": trainer.state.best_model_checkpoint, "epoch": float(trainer.state.epoch)})
        print(f"lr={rate:g}: val macro-F1 {trainer.state.best_metric:.4f}, epoch {trainer.state.epoch:.0f}", flush=True)
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
    pemenang = max(hasil_trial, key=lambda t: t["val_f1_macro"])
    model_sesi = AutoModelForSequenceClassification.from_pretrained(pemenang["checkpoint"], local_files_only=True)
    model_sesi.save_pretrained(MODEL_SESI_DIR)
    tokenizer.save_pretrained(MODEL_SESI_DIR)
    (MODEL_SESI_DIR / "trials.json").write_text(json.dumps(hasil_trial, indent=2), encoding="utf-8")
    display(pd.DataFrame(hasil_trial).style.format(precision=4).hide(axis="index"))
    print(f"Pemenang lr={pemenang['learning_rate']:g}; durasi {(perf_counter() - mulai) / 60:.1f} menit; disimpan di {MODEL_SESI_DIR}")
else:
    print("Training dilewati: LATIH_INDOBERT = False; model dimuat dari", MODEL_SESI_DIR)

Map:   0%|          | 0/7023 [00:00<?, ? examples/s]

Map:  57%|█████▋    | 4000/7023 [00:00<00:00, 16442.88 examples/s]

Map: 100%|██████████| 7023/7023 [00:00<00:00, 23942.83 examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

Map: 100%|██████████| 842/842 [00:00<00:00, 40811.28 examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (6 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.584000,0.313496,0.896675,0.881648
2,0.301100,0.327740,0.883610,0.871312
3,0.194400,0.359357,0.912114,0.898471
4,0.116400,0.530820,0.915677,0.906114
5,0.065100,0.598005,0.910926,0.896417
6,0.033600,0.640579,0.914489,0.901910
7,0.017900,0.697584,0.915677,0.903012


lr=3e-05: val macro-F1 0.9061, epoch 7


learning_rate,val_f1_macro,checkpoint,epoch
0.0000,0.9091,models\eksperimen\runs_intent_sesi\trial_1\checkpoint-1317,6.0000
0.0000,0.9036,models\eksperimen\runs_intent_sesi\trial_2\checkpoint-1756,7.0000
0.0000,0.9061,models\eksperimen\runs_intent_sesi\trial_3\checkpoint-1756,7.0000


Pemenang lr=1e-05; durasi 19.9 menit; disimpan di models\eksperimen\intent_sesi_transformer


## Evaluasi Ujung ke Ujung

Fungsi penilaian sama dengan `nlu_gabungan`: target disusun dari relasi kandidat dan disaring sesuai intent tebakan, ambang dipilih dari validation untuk tiap sistem, lalu exact match dihitung per pesan test.

| Sistem | Intent | Relasi |
| --- | --- | --- |
| Pipeline lama | Model 1 lama (`nlu_baru`) | Model 2 dengan awalan intent tebakan |
| **Pipeline + data sesi** | Model 1 eksperimen (notebook ini) | Model 2 dengan awalan intent tebakan |
| Gabungan | Kepala intent model gabungan | Kepala relasi tanpa awalan intent |

Setiap sistem dinilai dua kali: intent dari **teks mentah**, dan intent dari rata-rata **varian nama** (kondisi game).

In [6]:
AMBANG_KANDIDAT = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]


def susun_target(kandidat_relasi, intent):
    boleh = {"neutral"} if intent == "neutral" else {"offend", "defend"}
    targets = [{"pemain": k, "relasi": r} for k, r in kandidat_relasi if r in boleh]
    if targets or intent == "neutral":
        return targets
    return [{"pemain": "tidak_diketahui", "relasi": intent}]


def terapkan_ambang(pairs, prediksi, proba, intent_pesan, ambang):
    hasil = pd.Series(list(prediksi), index=pairs.index)
    if ambang <= 0:
        return hasil
    for row_id, grup in pairs.assign(prediksi=hasil.values).groupby("row_id"):
        intent = intent_pesan[row_id]
        if intent == "neutral" or grup["prediksi"].isin(["offend", "defend"]).any():
            continue
        peluang_intent = proba.loc[grup.index, intent]
        if peluang_intent.max() >= ambang:
            hasil[peluang_intent.idxmax()] = intent
    return hasil


def nilai_pesan(rows, pairs, prediksi, intent_pesan):
    pairs = pairs.assign(prediksi=list(prediksi))
    records = []
    for row_id, row in rows.iterrows():
        grup = pairs[pairs["row_id"] == row_id]
        got = {(t["pemain"], t["relasi"]) for t in susun_target(zip(grup["kandidat"], grup["prediksi"]), intent_pesan[row_id])}
        gold = {(t["pemain"], t["relasi"]) for t in row["target"]}
        records.append({"row_id": row_id, "intent_asli": row["label_intent"], "intent_prediksi": intent_pesan[row_id],
                        "benar_persis": gold == got})
    detail = pd.DataFrame(records)
    return {"exact_match": float(detail["benar_persis"].mean()),
            "intent_benar": float((detail["intent_asli"] == detail["intent_prediksi"]).mean()), "detail": detail}


def pilih_ambang(rows, pairs, proba, intent_pesan):
    mentah = proba.idxmax(axis=1)
    skor = [(a, nilai_pesan(rows, pairs, terapkan_ambang(pairs, mentah, proba, intent_pesan, a), intent_pesan)["exact_match"])
            for a in AMBANG_KANDIDAT]
    terbaik = max(e for _, e in skor)
    return max(a for a, e in skor if e == terbaik)


# Probabilitas kelas dari model klasifikasi; kolom memakai nama label model.
@torch.inference_mode()
def peluang_klasifikasi(tok, mdl, teks, max_length):
    keluar = []
    for k in range(0, len(teks), 16):
        enc = tok(list(teks[k:k + 16]), truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=PRESISI, enabled=PRESISI is not None):
            keluar.append(torch.softmax(mdl(**enc).logits.float(), dim=-1).cpu().numpy())
    return pd.DataFrame(np.concatenate(keluar), columns=[mdl.config.id2label[i] for i in range(mdl.config.num_labels)])


def muat_klasifikasi(folder):
    return (AutoTokenizer.from_pretrained(folder, local_files_only=True),
            AutoModelForSequenceClassification.from_pretrained(folder, local_files_only=True).to(DEVICE).eval())


# Intent satu pesan: teks mentah, atau rata-rata varian nama (kondisi game).
def intent_pesan(peluang_teks, rows, varian):
    return {row_id: INTENT_LABELS[int(peluang_teks(
        varian_nama_intent(row["teks_chat"], row["daftar_pemain"], COMMON_WORDS) if varian else [row["teks_chat"]]
    ).mean(axis=0).argmax())] for row_id, row in rows.iterrows()}


# Satu sistem: ambang dari validation, exact match di test.
def nilai_sistem(intent_bagian, pairs_bagian, proba_bagian):
    ambang = pilih_ambang(split_target["validation"], pairs_bagian["validation"], proba_bagian["validation"],
                          intent_bagian["validation"])
    pairs, proba = pairs_bagian["test"], proba_bagian["test"]
    prediksi = terapkan_ambang(pairs, proba.idxmax(axis=1), proba, intent_bagian["test"], ambang)
    return {"ambang": ambang, **nilai_pesan(split_target["test"], pairs, prediksi, intent_bagian["test"])}


tok2, model_2 = muat_klasifikasi(MODEL_2_DIR)


# Pipeline: pasangan dengan awalan intent tebakan lalu relasi Model 2.
def nilai_pipeline(peluang_teks, varian):
    intent_bagian = {n: intent_pesan(peluang_teks, split_target[n], varian) for n in ("validation", "test")}
    pairs = {n: bentuk_pasangan(split_target[n], COMMON_WORDS, JUMLAH_PEMAIN_MINIMUM, intent_bagian[n]) for n in intent_bagian}
    proba = {n: peluang_klasifikasi(tok2, model_2, f["teks"].tolist(), 512).set_index(f.index) for n, f in pairs.items()}
    return nilai_sistem(intent_bagian, pairs, proba)


hasil = {}
for nama, folder in (("Pipeline lama", MODEL_1_DIR), ("Pipeline + data sesi", MODEL_SESI_DIR)):
    tok1, model_1 = muat_klasifikasi(folder)
    peluang_1 = lambda teks: peluang_klasifikasi(tok1, model_1, list(teks), 128)[INTENT_LABELS].to_numpy()
    prediksi_test = [INTENT_LABELS[i] for i in peluang_1(x_test).argmax(axis=1)]
    hasil[nama] = {"intent_acc": accuracy_score(y_test, prediksi_test),
                   "intent_f1": f1_score(y_test, prediksi_test, average="macro"),
                   "mentah": nilai_pipeline(peluang_1, False), "game": nilai_pipeline(peluang_1, True)}
    print(nama, "selesai dinilai", flush=True)
    del model_1

Pipeline lama selesai dinilai


Pipeline + data sesi selesai dinilai


### Model Gabungan dengan Cara Penilaian yang Sama

Model gabungan dimuat dari `nlu_gabungan` dan dinilai dengan fungsi yang sama, supaya ketiga sistem bisa dibandingkan dalam satu tabel.

In [7]:
# Arsitektur model gabungan (sama dengan nlu_gabungan) untuk memuat bobotnya.
class ModelGabungan(nn.Module):
    def __init__(self, encoder, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.kepala_intent = nn.Linear(encoder.config.hidden_size, len(INTENT_LABELS))
        self.kepala_relasi = nn.Linear(encoder.config.hidden_size, len(RELASI_LABELS))

    def forward(self, input_ids, attention_mask, token_type_ids=None, tugas="intent"):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).pooler_output
        return (self.kepala_intent if tugas == "intent" else self.kepala_relasi)(self.dropout(pooled))


model_g = ModelGabungan(AutoModel.from_config(AutoConfig.from_pretrained(MODEL_GABUNGAN_DIR, local_files_only=True)))
model_g.load_state_dict(load_file(str(MODEL_GABUNGAN_DIR / "model.safetensors")))
model_g.to(DEVICE).eval()
tok_g = AutoTokenizer.from_pretrained(MODEL_GABUNGAN_DIR, local_files_only=True)


@torch.inference_mode()
def peluang_gabungan(teks, tugas):
    keluar = []
    for k in range(0, len(teks), 16):
        enc = tok_g(list(teks[k:k + 16]), truncation=True, max_length=128 if tugas == "intent" else 512, padding=True,
                    return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=PRESISI, enabled=PRESISI is not None):
            keluar.append(torch.softmax(model_g(**enc, tugas=tugas).float(), dim=-1).cpu().numpy())
    return np.concatenate(keluar)


pasangan_g = {n: bentuk_pasangan(split_target[n], COMMON_WORDS, JUMLAH_PEMAIN_MINIMUM) for n in ("validation", "test")}
proba_g = {n: pd.DataFrame(peluang_gabungan(f["teks"].tolist(), "relasi"), columns=RELASI_LABELS, index=f.index)
           for n, f in pasangan_g.items()}
peluang_ig = lambda teks: peluang_gabungan(list(teks), "intent")
prediksi_test = [INTENT_LABELS[i] for i in peluang_ig(x_test).argmax(axis=1)]
hasil["Gabungan"] = {"intent_acc": accuracy_score(y_test, prediksi_test), "intent_f1": f1_score(y_test, prediksi_test, average="macro")}
for mode, varian in (("mentah", False), ("game", True)):
    intent_bagian = {n: intent_pesan(peluang_ig, split_target[n], varian) for n in ("validation", "test")}
    hasil["Gabungan"][mode] = nilai_sistem(intent_bagian, pasangan_g, proba_g)
print("Gabungan selesai dinilai")

Gabungan selesai dinilai


### Tabel Perbandingan

| Kolom | Arti |
| --- | --- |
| `Intent acc / macro-F1` | Pada test intent `nlu_baru` (1.123 chat, tanpa konteks). |
| `Intent pesan target` | Intent benar pada 155 pesan utama test target. |
| `Exact match` | Semua target satu pesan benar persis, ujung ke ujung. |
| `EM offend / defend / neutral` | Exact match per intent asli pesan. |

In [8]:
# Satu baris per sistem dan mode.
def baris(nama, h, mode):
    m = h[mode]
    per_intent = m["detail"].groupby("intent_asli")["benar_persis"].mean()
    return {"Sistem": nama, "Mode intent": "varian nama (game)" if mode == "game" else "teks mentah",
            "Intent acc": h["intent_acc"], "Intent macro-F1": h["intent_f1"], "Intent pesan target": m["intent_benar"],
            "Exact match": m["exact_match"], **{f"EM {k}": per_intent.get(k) for k in ("offend", "defend", "neutral")},
            "Ambang": m["ambang"]}


tabel = pd.DataFrame([baris(nama, h, mode) for mode in ("mentah", "game") for nama, h in hasil.items()])
display(tabel.style.format(precision=4).hide(axis="index"))
MODEL_SESI_DIR.mkdir(parents=True, exist_ok=True)
tabel.to_csv(MODEL_SESI_DIR / "perbandingan_tiga_sistem.csv", index=False)

Sistem,Mode intent,Intent acc,Intent macro-F1,Intent pesan target,Exact match,EM offend,EM defend,EM neutral,Ambang
Pipeline lama,teks mentah,0.9012,0.8837,0.5742,0.4968,0.7419,0.2059,0.6800,0.5000
Pipeline + data sesi,teks mentah,0.9012,0.8821,0.9484,0.8323,0.8065,0.8529,0.8400,0.5000
Gabungan,teks mentah,0.9065,0.8916,0.7097,0.7613,0.7742,0.8382,0.5200,0.5000
Pipeline lama,varian nama (game),0.9012,0.8837,0.5742,0.4968,0.7419,0.2059,0.6800,0.5000
Pipeline + data sesi,varian nama (game),0.9012,0.8821,0.9419,0.8258,0.8065,0.8382,0.8400,0.5000
Gabungan,varian nama (game),0.9065,0.8916,0.6968,0.7613,0.7742,0.8382,0.5200,0.5000


## Catatan Evaluasi

- Data tambahan plan03 tidak dipakai, sehingga hasil setara dengan run 27 Oktober.
- Pesan sesi diambil hanya dari sesi train target. Test intent dan test target tidak pernah dipakai untuk training.
- Model eksperimen disimpan terpisah (`models/eksperimen/`) dan **tidak** dipakai game.
- Test target hanya 155 pesan: selisih 1 pesan ≈ 0,6 poin exact match.